# Indian Cattle Breed Classifier — Kaggle Training Notebook

Transfer learning with a **timm** backbone (EfficientNet-B0 by default) on a
folder-per-breed image dataset.

**Setup on Kaggle**
1. Add your dataset (Add Input → your uploaded cattle breeds dataset).
2. Set `DATA_DIR` below to the folder that contains one sub-folder per breed.
3. Turn on GPU (Settings → Accelerator → GPU T4/P100).
4. Run all cells. The best model + plots are written to `/kaggle/working/`.


## 1. Install & imports

In [ ]:
# timm is usually preinstalled on Kaggle; install if missing.
import importlib, subprocess, sys
if importlib.util.find_spec("timm") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"])

import os, json, time, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import timm
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| timm", timm.__version__)

## 2. Config
Edit `DATA_DIR` to point at your breed folders.

In [ ]:
CONFIG = {
    # >>> Path to the folder that CONTAINS train/ val/ test/ sub-folders <<<
    # On Kaggle, after adding your dataset it will look like:
    #   /kaggle/input/<your-dataset-name>
    "data_dir": "/kaggle/input/your-cattle-dataset",
    "has_split": True,               # your data is already split into train/val/test
    "image_size": 224,
    # used only if has_split is False (single folder-per-breed dir):
    "val_split": 0.15,
    "test_split": 0.10,
    "backbone": "efficientnet_b0",   # try: resnet50, convnext_tiny, mobilenetv3_large_100
    "pretrained": True,
    "dropout": 0.2,
    "epochs": 25,
    "batch_size": 32,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "label_smoothing": 0.1,
    "freeze_epochs": 3,
    "early_stopping_patience": 6,
    "num_workers": 2,
    "seed": 42,
    "out_dir": "/kaggle/working",
}

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(CONFIG["seed"])
assert os.path.isdir(CONFIG["data_dir"]), f"data_dir not found: {CONFIG['data_dir']}"

## 3. Data: transforms, stratified split, loaders

In [ ]:
MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
size = CONFIG["image_size"]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(size, scale=(0.7,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2,0.2,0.2,0.05),
    transforms.ToTensor(), transforms.Normalize(MEAN,STD),
    transforms.RandomErasing(p=0.25),
])
eval_tf = transforms.Compose([
    transforms.Resize(int(size*1.14)), transforms.CenterCrop(size),
    transforms.ToTensor(), transforms.Normalize(MEAN,STD),
])

def align_to_master(ds, master):
    """Lock a split's labels to the train class order (handles a breed that is
    missing from a split, e.g. test/ having fewer breeds than train/)."""
    remap={ds.class_to_idx[c]:master[c] for c in ds.classes}
    ds.samples=[(p,remap[t]) for p,t in ds.samples]
    ds.targets=[remap[t] for t in ds.targets]; ds.imgs=ds.samples
    return ds

if CONFIG["has_split"]:
    train_ds = datasets.ImageFolder(os.path.join(CONFIG["data_dir"],"train"), train_tf)
    classes = train_ds.classes
    master = train_ds.class_to_idx
    val_ds = align_to_master(datasets.ImageFolder(os.path.join(CONFIG["data_dir"],"val"), eval_tf), master)
    test_dir = os.path.join(CONFIG["data_dir"],"test")
    test_ds = align_to_master(datasets.ImageFolder(test_dir, eval_tf), master) if os.path.isdir(test_dir) else val_ds
else:
    base = datasets.ImageFolder(CONFIG["data_dir"]); classes = base.classes
    def stratified(targets, vf, tf_, seed):
        rng=np.random.default_rng(seed); targets=np.array(targets); tr,va,te=[],[],[]
        for c in np.unique(targets):
            idx=np.where(targets==c)[0]; rng.shuffle(idx); n=len(idx)
            nt=int(round(n*tf_)); nv=int(round(n*vf))
            te+=idx[:nt].tolist(); va+=idx[nt:nt+nv].tolist(); tr+=idx[nt+nv:].tolist()
        return tr,va,te
    tr_idx,va_idx,te_idx=stratified(base.targets,CONFIG["val_split"],CONFIG["test_split"],CONFIG["seed"])
    train_ds=Subset(datasets.ImageFolder(CONFIG["data_dir"],train_tf),tr_idx)
    eb=datasets.ImageFolder(CONFIG["data_dir"],eval_tf)
    val_ds=Subset(eb,va_idx); test_ds=Subset(eb,te_idx)

print(f"{len(classes)} breeds")
print(f"train {len(train_ds)} | val {len(val_ds)} | test {len(test_ds)}")

pin = torch.cuda.is_available(); nw=CONFIG["num_workers"]; bs=CONFIG["batch_size"]
train_loader=DataLoader(train_ds,bs,shuffle=True,num_workers=nw,pin_memory=pin)
val_loader=DataLoader(val_ds,bs,shuffle=False,num_workers=nw,pin_memory=pin)
test_loader=DataLoader(test_ds,bs,shuffle=False,num_workers=nw,pin_memory=pin)

with open(os.path.join(CONFIG["out_dir"],"label_map.json"),"w") as f:
    json.dump({i:c for i,c in enumerate(classes)}, f, indent=2, ensure_ascii=False)

## 4. Model

In [ ]:
class CattleBreedClassifier(nn.Module):
    def __init__(self, backbone, num_classes, pretrained=True, dropout=0.2):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=pretrained,
                                           num_classes=0, global_pool="avg")
        self.head = nn.Sequential(nn.Dropout(dropout),
                                  nn.Linear(self.backbone.num_features, num_classes))
    def forward(self,x): return self.head(self.backbone(x))
    def set_backbone_trainable(self, t):
        for p in self.backbone.parameters(): p.requires_grad = t

model = CattleBreedClassifier(CONFIG["backbone"], len(classes),
                              CONFIG["pretrained"], CONFIG["dropout"]).to(device)
print(sum(p.numel() for p in model.parameters())/1e6, "M params")

## 5. Train

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"],
                              weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"])
use_amp = device.type=="cuda"
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

def run_epoch(loader, train):
    model.train(train)
    tot_loss=tot_acc=n=0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs,y in tqdm(loader, leave=False):
            imgs,y = imgs.to(device,non_blocking=True), y.to(device,non_blocking=True)
            if train: optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                out=model(imgs); loss=criterion(out,y)
            if train:
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            b=imgs.size(0); n+=b
            tot_loss+=loss.item()*b
            tot_acc+=(out.argmax(1)==y).float().sum().item()
    return tot_loss/n, tot_acc/n

if CONFIG["freeze_epochs"]>0:
    model.set_backbone_trainable(False); print("Backbone frozen.")

hist={"train_loss":[],"train_acc":[],"val_loss":[],"val_acc":[]}
best=0.0; bad=0; ckpt_path=os.path.join(CONFIG["out_dir"],"best_model.pt")
for ep in range(1, CONFIG["epochs"]+1):
    if CONFIG["freeze_epochs"]>0 and ep==CONFIG["freeze_epochs"]+1:
        model.set_backbone_trainable(True); print("Backbone unfrozen.")
    t0=time.time()
    tl,ta=run_epoch(train_loader,True); vl,va=run_epoch(val_loader,False)
    scheduler.step()
    for k,v in zip(hist,[tl,ta,vl,va]): hist[k].append(v)
    print(f"Ep {ep:02d} | train {tl:.3f}/{ta:.3f} | val {vl:.3f}/{va:.3f} | {time.time()-t0:.0f}s")
    if va>best:
        best=va; bad=0
        torch.save({"model_state":model.state_dict(),"classes":classes,
                    "backbone":CONFIG["backbone"],"image_size":CONFIG["image_size"],
                    "val_acc":va,"epoch":ep}, ckpt_path)
        print(f"  saved best (val_acc={va:.3f})")
    else:
        bad+=1
        if bad>=CONFIG["early_stopping_patience"]:
            print("Early stopping."); break
json.dump(hist, open(os.path.join(CONFIG["out_dir"],"history.json"),"w"), indent=2)
print("Best val acc:", round(best,4))

## 6. Evaluate on the held-out test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state"]); model.eval()

ys,ps=[],[]
with torch.no_grad():
    for imgs,y in test_loader:
        out=model(imgs.to(device)); ps.append(out.argmax(1).cpu().numpy()); ys.append(y.numpy())
y_true=np.concatenate(ys); y_pred=np.concatenate(ps)
print("Test accuracy:", round(accuracy_score(y_true,y_pred),4))
print(classification_report(y_true,y_pred,target_names=classes,digits=4,zero_division=0))

## 7. Plots: training curves & confusion matrix

In [ ]:
ep=range(1,len(hist["train_loss"])+1)
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].plot(ep,hist["train_loss"],label="train"); ax[0].plot(ep,hist["val_loss"],label="val")
ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(ep,hist["train_acc"],label="train"); ax[1].plot(ep,hist["val_acc"],label="val")
ax[1].set_title("Accuracy"); ax[1].legend()
fig.tight_layout(); fig.savefig(os.path.join(CONFIG["out_dir"],"training_curves.png"),dpi=120); plt.show()

cm=confusion_matrix(y_true,y_pred,labels=range(len(classes)))
fig,ax=plt.subplots(figsize=(max(6,len(classes)*0.6),max(5,len(classes)*0.6)))
im=ax.imshow(cm,cmap="Blues")
ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes,rotation=90)
ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion matrix")
th=cm.max()/2 if cm.max() else 0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j,i,cm[i,j],ha="center",va="center",
                color="white" if cm[i,j]>th else "black",fontsize=7)
fig.colorbar(im,ax=ax,fraction=0.046); fig.tight_layout()
fig.savefig(os.path.join(CONFIG["out_dir"],"confusion_matrix.png"),dpi=120); plt.show()

## 8. Predict on a single image

In [ ]:
from PIL import Image
def predict(path, topk=3):
    img=Image.open(path).convert("RGB")
    x=eval_tf(img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs=F.softmax(model(x),1)[0]
    conf,idx=probs.topk(min(topk,len(classes)))
    return [(classes[i],float(c)) for c,i in zip(conf,idx)]

# Example (point at any test image):
# print(predict("/kaggle/input/your-cattle-dataset/Gir/xxx.jpg"))